In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving trainV2.csv to trainV2.csv
Saving test_with_labels.csv to test_with_labels.csv


In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

In [ ]:
train_df = pd.read_csv("trainV2.csv")
test_df = pd.read_csv("test_with_labels.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (3652, 2)
Test shape: (913, 2)


,Text,Class
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,Non-Abusive
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,Abusive
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,Non-Abusive
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,Abusive
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,Non-Abusive


In [ ]:
def clean_label(x):
    x = str(x).strip().lower()
    if x == "non-abusive":
        return 1
    else:
        return 0

train_df["Class"] = train_df["Class"].apply(clean_label)
test_df["Class"] = test_df["Class"].apply(clean_label)

print(train_df["Class"].value_counts())

Class
1    1883
0    1769
Name: count, dtype: int64


In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["Text"],
    train_df["Class"],
    test_size=0.1,
    random_state=42,
    stratify=train_df["Class"]
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

Train size: 3286
Validation size: 366


In [ ]:
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [ ]:
def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_df["Text"])

In [ ]:
train_dataset = Dataset.from_dict({
    **train_encodings,
    "labels": list(train_labels)
})

val_dataset = Dataset.from_dict({
    **val_encodings,
    "labels": list(val_labels)
})

test_dataset = Dataset.from_dict({
    **test_encodings,
    "labels": list(test_df["Class"])
})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)

    return {
        "accuracy": float(f"{acc:.4f}"),
        "precision": float(f"{prec:.4f}"),
        "recall": float(f"{rec:.4f}"),
        "f1": float(f"{f1:.4f}")
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    logging_dir="./logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=False,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.500696


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=618, training_loss=0.46893666708739445, metrics={'train_runtime': 188.0475, 'train_samples_per_second': 52.423, 'train_steps_per_second': 3.286, 'total_flos': 648437195934720.0, 'train_loss': 0.46893666708739445, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("\nFinal Test Results:")
print("Accuracy :", format(test_results["eval_accuracy"], ".4f"))
print("Precision:", format(test_results["eval_precision"], ".4f"))
print("Recall   :", format(test_results["eval_recall"], ".4f"))
print("F1 Score :", format(test_results["eval_f1"], ".4f"))


Final Test Results:
Accuracy : 0.7919
Precision: 0.8294
Recall   : 0.7521
F1 Score : 0.7889
